# elbo-loss-sum-with-beta — ex1: beta-weighted ELBO sum + sweep plot

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `elbo-loss-sum-with-beta`. Running the final beacon cell reports progress against the `VAE: ELBO loss sum with beta` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: ELBO loss sum with beta` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`elbo-loss-sum-with-beta`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "elbo-loss-sum-with-beta"
DD_SUBTOPIC = "VAE: ELBO loss sum with beta"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ELBO loss (sum with beta) — quick refresher

The VAE training objective decomposes into TWO terms:

```
loss = reconstruction + beta * kl
```

- `reconstruction` — e.g. `F.mse_loss(decoded, original)` or a Bernoulli cross-entropy. Drives the decoder to actually decode.
- `kl` — the closed-form Gaussian KL term. Pulls the latent distribution toward the standard-normal prior.
- `beta` — a positive scalar trading reconstruction quality against latent regularity.

**`beta = 1` is the vanilla VAE.** Maximizes the true ELBO. **`beta > 1` is the beta-VAE** (Higgins et al. 2017) — encourages DISENTANGLED latents at the cost of reconstruction. **`beta < 1` tilts toward reconstruction** at the cost of latent regularity — useful when you don't care about sampling from the prior.

**Both terms are SCALARS before you combine them.** Reduce each to a 0-D tensor (mean or sum over batch+dims), THEN add. Adding a `(B,)` to a `()` will broadcast — silent bug.

### Exercise 1 — beta-weighted ELBO sum + sweep plot

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the beta-weighted ELBO recipe `loss = reconstruction + beta * kl` to combine two pre-reduced scalar loss terms into a single scalar training loss.
> Keywords: elbo, beta-vae, loss-composition, scalar
> ```

**KCs targeted:** `elbo-sum-recon-plus-kl`, `beta-weighting`

Implement `ex1_elbo_loss(reconstruction, kl, beta)`. The one-line VAE loss combinator:

1. `reconstruction` is a 0-D scalar tensor — the pre-reduced reconstruction loss (e.g. `F.mse_loss(decoded, original)`).
2. `kl` is a 0-D scalar tensor — the batch-mean Gaussian KL.
3. `beta` is a Python `float` — the KL weighting.
4. Return `reconstruction + beta * kl` as a 0-D scalar tensor.

Input: two scalar tensors + a float.
Output: scalar tensor.

The visualization sweeps `beta` from 0 to 4 on a fixed (reconstruction, kl) pair and plots how the composite loss scales — the slope IS `kl`, the intercept IS `reconstruction`.

In [ ]:
def ex1_elbo_loss(reconstruction: Tensor, kl: Tensor, beta: float) -> Tensor:
    return reconstruction + beta * kl


<details><summary>Solution</summary>

```python
def ex1_elbo_loss(reconstruction: Tensor, kl: Tensor, beta: float) -> Tensor:
    return reconstruction + beta * kl
```

**Why `beta` is a float, not a tensor.** `beta` is a fixed hyperparameter — not a learned quantity. Passing it as a Python float saves wrapping/unwrapping and makes the call site read naturally: `loss = elbo(recon, kl, beta=4.0)`.

**The two terms are already scalar.** Make sure your `reconstruction` and `kl` inputs are 0-D tensors BEFORE calling this. If `kl` were `(B,)`, broadcasting would add `kl` to `recon` elementwise and you'd silently get a `(B,)` 'loss' — which `.backward()` would accept (autograd sums it implicitly), but the per-sample average is wrong.

**`beta=1` IS the true ELBO.** Anything else trades regularity for reconstruction (or vice versa) — useful, but no longer an ELBO. Higgins et al. (2017) showed `beta > 1` encourages disentangled latent factors on visual datasets.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()